In [5]:
# !pip install tensorflow==2.16.1
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time

os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict


METADATA_FILE = '_METADATA'
_CHECKPOINT_FILE = 'checkpoint'

import nest_asyncio
nest_asyncio.apply()

def load_model(read_dir):
    read_dir = epath.Path(read_dir) # must be epath.Path object
    metadata_path = read_dir / METADATA_FILE
    back_metadata_path = read_dir / f'{METADATA_FILE}.back'
    try:
        metadata_path.rename(back_metadata_path)
    except:
        pass
    metadata_path.unlink(missing_ok=True) # delete
    structure_path = read_dir / _CHECKPOINT_FILE
    msgpack = ocp.aggregate_handlers.MsgpackHandler(0)
    structure = msgpack.deserialize(structure_path)
    # backup original checkpoint fil
    back_structure_path = read_dir / 'checkpoint_back'
    back_structure = structure.copy()
    if not back_structure_path.exists():
        asyncio.run(msgpack.serialize(back_structure_path, item=back_structure))
    print(f'Old structure file keys: {structure.keys()}')
    remove_keys = ['opt_state', 'step'] # select the weight name you don't want to load, all weight name: opt_state, step, params
    _ = [structure.pop(key) for key in remove_keys if key in structure]
    print(f'New structure file keys: {structure.keys()}')
    asyncio.run(msgpack.serialize(structure_path, item=structure))  # rewrite struct file

    # load model based struct, note: axes must same as training
    mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'autoregressive']
    devices = np.asarray(jax.devices()).reshape([1] * len(mesh_axes))
    mesh = jax.sharding.Mesh(devices, mesh_axes)
    sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
    weight_dtype = np.float32 # set restore weights dtype, np.float32 or np.float16
    restore_args = {}
    for k, v in flatten_dict(structure).items():
        restore_args[k] =  ocp.ArrayRestoreArgs(restore_type=jax.Array, dtype=weight_dtype, sharding=sharding)
    restore_args = unflatten_dict(restore_args)
    ckptr = ocp.Checkpointer(ocp.PyTreeCheckpointHandler())
    w = ckptr.restore(read_dir, args=ocp.args.PyTreeRestore(restore_args=restore_args))
    structure_path = read_dir / _CHECKPOINT_FILE
    # rewrite struct file, otherwise occur error when continue training
    asyncio.run(msgpack.serialize(structure_path, item=back_structure))
    while 'params' in w:
        w = w['params']
    flat_w = {'.'.join(k): np.array(v) for k, v in flatten_dict(w).items()}
    try:
        back_metadata_path.rename(metadata_path)
    except:
        pass
    return flat_w


In [6]:
# model_path = 'gs://llm_base_models_europe-west4/v5p_256/7B/xm_E8S0T2A11B_CopyInit_0102/checkpoints/151050/state'
model_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/xm_E8S0T2A11B_CopyInit_0102/checkpoints/151050/state'
flat_w = load_model(model_path)

Old structure file keys: dict_keys(['params'])
New structure file keys: dict_keys(['params'])


/tmp/ipykernel_163894/338267694.py:47: RuntimeWarning: coroutine 'MsgpackHandler.serialize' was never awaited
  msgpack.serialize(structure_path, item=structure)  # rewrite struct file
I0613 08:02:01.779257  165318 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com
/tmp/ipykernel_163894/338267694.py:63: RuntimeWarning: coroutine 'MsgpackHandler.serialize' was never awaited
  msgpack.serialize(structure_path, item=back_structure)


In [ ]:
import re

# scan_layers=False
new_params = {}
for key, v in flat_w.items():
    # key = 'decoder.layers.self_attention_0.AttentionOp_0.dyn_w_proj.dw1.kernel'
    lyx = re.findall('_(\d+)', key)
    key = key.replace('AttentionOp_0', 'attention_op')
    if lyx:
        lyx = int(lyx[0])
        print(lyx)
        new_key = key.replace(f'_{lyx}.', '.').replace('unshared_mlp', 'moe').replace('router_gate', 'gate').  \
        replace('mgate', 'mgate.kernel').replace('k_norm', 'qk_norm.k_norm').replace('q_norm', 'qk_norm.q_norm')
        base_key = new_key.replace('decoder.layers.', f'params.decoder.layers_{lyx}.sub_0.')
        print(new_key)
        for i in range(lyx, 48, 4):
            new_key = base_key.replace(f'layers_{lyx}', f'layers_{i}')
            print(f'new_key: {new_key}')
            new_v = v[:, i // 4]
            new_params[tuple(new_key.split('.'))] = new_v
    else:
        key = 'params.' + key
        new_params[tuple(key.split('.'))] = v.astype(jnp.bfloat16)
    print('\n\n\n')
    
# scan_layers=True
# import re
# import jax.numpy as jnp

# new_params = {}
# for key, v in flat_w.items():
#     # key = 'decoder.layers.self_attention_0.AttentionOp_0.dyn_w_proj.dw1.kernel'
#     lyx = re.findall('_(\d+)', key)
#     key = key.replace('AttentionOp_0', 'attention_op')
#     if lyx:
#         lyx = int(lyx[0])
#         print(lyx)
#         new_key = key.replace(f'_{lyx}.', '.').replace('unshared_mlp', 'moe').replace('router_gate', 'gate').  \
#         replace('mgate', 'mgate.kernel').replace('k_norm', 'qk_norm.k_norm').replace('q_norm', 'qk_norm.q_norm')
#         new_key = new_key.replace('decoder.layers.', f'params.decoder.layers.sub_{lyx}.')
#         print(new_key)
#         new_params[tuple(new_key.split('.'))] = v.astype(jnp.bfloat16)
#     else:
#         key = 'params.' + key
#         new_params[tuple(key.split('.'))] = v.astype(jnp.bfloat16)
#     print('\n\n\n')
    





0
decoder.layers.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_0.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_4.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_8.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_12.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_16.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_20.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_24.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_28.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_32.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_36.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_40.sub_0.post_self_attention_layer_norm.scale
new_key: params.decoder.layers_44.sub_0.post_self_attention_layer_norm.scale




1
decoder.layers.

In [52]:
sorted_new_params = sorted(new_params.items(), key=lambda x: x[0])

for k, v in sorted_new_params:
    print(k, v.shape)

('decoder', 'decoder_norm', 'scale') (4096,)
('decoder', 'logits_dense', 'kernel') (4096, 152064)
('params', 'decoder', 'layers_0', 'sub_0', 'moe', 'gate', 'kernel') (4096, 8)
('params', 'decoder', 'layers_0', 'sub_0', 'moe', 'mgate', 'kernel') (8, 4096, 44)
('params', 'decoder', 'layers_0', 'sub_0', 'moe', 'wi_0') (8, 4096, 5632)
('params', 'decoder', 'layers_0', 'sub_0', 'moe', 'wi_1') (8, 4096, 5632)
('params', 'decoder', 'layers_0', 'sub_0', 'moe', 'wo') (8, 5632, 4096)
('params', 'decoder', 'layers_0', 'sub_0', 'post_self_attention_layer_norm', 'scale') (4096,)
('params', 'decoder', 'layers_0', 'sub_0', 'pre_self_attention_layer_norm', 'scale') (4096,)
('params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'attention_op', 'dyn_w_proj', 'dd', 'kernel') (4096, 1, 128)
('params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'attention_op', 'dyn_w_proj', 'dw1', 'kernel') (4096, 1, 4, 128)
('params', 'decoder', 'layers_0', 'sub_0', 'self_attention', 'attention_op', 'dyn_w_pr

In [ ]:
import orbax.checkpoint as ocp

unflated_params = unflatten_dict(new_params)
# save model
checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/xmv4.0/DC.Moe.A11.E8T2.32k/DCLlama7BOpenMoe/checkpoints/0/items'
orbax_checkpointer = ocp.PyTreeCheckpointer()
orbax_checkpointer.save(checkpoint_dir, {"params": unflated_params}, force=True)
print(f"Quantized params checkpoint saved at: {checkpoint_dir}")

In [8]:
for k, v in flat_w.items():
    print(k, v.shape)

decoder.decoder_norm.scale (4096,)
decoder.layers.post_self_attention_layer_norm_0.scale (4096, 12)
decoder.layers.post_self_attention_layer_norm_1.scale (4096, 12)
decoder.layers.post_self_attention_layer_norm_2.scale (4096, 12)
decoder.layers.post_self_attention_layer_norm_3.scale (4096, 12)
decoder.layers.pre_self_attention_layer_norm_0.scale (4096, 12)
decoder.layers.pre_self_attention_layer_norm_1.scale (4096, 12)
decoder.layers.pre_self_attention_layer_norm_2.scale (4096, 12)
decoder.layers.pre_self_attention_layer_norm_3.scale (4096, 12)
decoder.layers.self_attention_0.AttentionOp_0.dyn_w_proj.dd.kernel (4096, 12, 1, 128)
decoder.layers.self_attention_0.AttentionOp_0.dyn_w_proj.dw1.kernel (4096, 12, 1, 4, 128)
decoder.layers.self_attention_0.AttentionOp_0.dyn_w_proj.qkw (1, 12, 4, 128, 4, 32)
decoder.layers.self_attention_0.k_norm.scale (128, 12)
decoder.layers.self_attention_0.key.kernel (4096, 12, 32, 128)
decoder.layers.self_attention_0.out.kernel (32, 12, 128, 4096)
decoder.